In [18]:
import numpy as np
import rasterio as rio
from exactextract import __version__

print(__version__)

0.2.0.dev0


In [19]:
SICK_FILE = "../data/processed/population_fix.tif"
OK_FILE = "../data/processed/population_f64.tif"
with rio.open(SICK_FILE) as src:
    print("===SICK===")
    print(src.profile)
with rio.open(OK_FILE) as src:
    print("===OK===")
    print(src.profile)

===SICK===
{'driver': 'GTiff', 'dtype': 'int32', 'nodata': -200.0, 'width': 15657, 'height': 13426, 'count': 1, 'crs': CRS.from_epsg(4326), 'transform': Affine(0.00227456, 0.0, -79.61249969999994,
       0.0, -0.00227456, 10.059076038000043), 'blockysize': 1, 'tiled': False, 'compress': 'deflate', 'interleave': 'band'}
===OK===
{'driver': 'GTiff', 'dtype': 'float32', 'nodata': -200.0, 'width': 15657, 'height': 13426, 'count': 1, 'crs': CRS.from_epsg(4326), 'transform': Affine(0.00227456, 0.0, -79.61249969999994,
       0.0, -0.00227456, 10.059076038000043), 'blockysize': 1, 'tiled': False, 'compress': 'deflate', 'interleave': 'band'}


In [20]:
from exactextract import exact_extract

SICK_FILE = "../data/processed/population_fix.tif"

exact_extract(SICK_FILE, "../data/processed/test_zone.json", ops=["sum"])

RuntimeError: Unable to cast Python instance of type <class 'float'> to C++ type '?' (#define PYBIND11_DETAILED_ERROR_MESSAGES or compile in debug mode for details)

In [37]:
# Rewrite sick file to int64
with rio.open("../data/processed/population.tif") as src:
    data = src.read(1)
    profile: dict = src.profile.copy()
    profile.update(dtype="int64")
    # profile.update(nodata=0)
    with rio.open("../data/processed/population_i64.tif", "w", **profile) as dst:
        dst.write(np.where(data == src.nodata, src.nodata, data.astype(np.int64)), 1)

In [38]:
with rio.open("../data/processed/population_i64.tif") as src:
    print(src.profile)

exact_extract("../data/processed/population_i64.tif", "../data/processed/test_zone.json", ops=["sum"])

{'driver': 'GTiff', 'dtype': 'int64', 'nodata': -2147483647.0, 'width': 21368, 'height': 18324, 'count': 1, 'crs': CRS.from_epsg(4326), 'transform': Affine(0.0016666666600653845, 0.0, -79.61269403265335,
       0.0, -0.0016666666600653845, 10.06065883749271), 'blockysize': 1, 'tiled': False, 'compress': 'deflate', 'interleave': 'band'}


RuntimeError: Unable to cast Python instance of type <class 'float'> to C++ type '?' (#define PYBIND11_DETAILED_ERROR_MESSAGES or compile in debug mode for details)

In [28]:
import json

import numpy as np
import rasterio as rio
from exactextract import __version__, exact_extract

print("exact_exctract version: ", __version__)

GEOJSON = {
    "type": "Feature",
    "properties": {},
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [3.0, 7.0],
                [3.0, 10.0],
                [0.0, 10.0],
                [0.0, 7.0],
                [3.0, 7.0],
            ],
        ],
    },
}

with open("test_zone.json", "w") as out:
    json.dump(GEOJSON, out)

data = np.array([[0, 1, 0], [1, 9, 1], [0, 1, 0]])
transform = rio.transform.from_origin(0, 10, 1, 1)
with rio.open(
    "sick_raster.tif",
    "w",
    driver="GTiff",
    width=data.shape[1],
    height=data.shape[0],
    count=1,
    dtype="uint8",
    crs="+proj=latlong",
    transform=transform,
    nodata=2,
) as dst:
    dst.write(data, 1)

exact_extract("sick_raster.tif", "test_zone.json", ops=["sum"])

exact_exctract version:  0.2.0.dev0


[{'type': 'Feature', 'properties': {'sum': 13.0}}]